# Actividad 04 — Escalado (StandardScaler fit solo en train)

**Fase:** 2 — Ingeniería de Características Multimodal  
**Dominio:** Predicción de producción de limón (Sutil y Dulce), 2016-2025  
**Referencia metodológica:** v1 `notebooks/fase2/actividad_04_escalado_normalizacion.ipynb`
(split cronológico, scaler ajustado exclusivamente sobre train)

---

## Objetivo

1. **Ajustar StandardScaler SOLO sobre el conjunto de entrenamiento** (2016-07 a 2023-12,
   90 meses) para todas las columnas numéricas — producción, precio, clima, lags, INDECI y NLP.
2. **Transformar val y test con el scaler ajustado en train** (nunca fit sobre val/test).
3. **Excluir del escalado** `mes_sin`/`mes_cos` (ya en [-1,1] por diseño) y las claves
   deseadas `año`/`mes` (identificadores de fila, no features de entrada).
4. **Guardar el scaler (joblib)** para desnormalizar predicciones.
5. **Verificar**: media≈0 y desv≈1 en train; val/test NO media exactamente 0.

## Entrada

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2_features.csv` (114×32)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2_features.csv` (114×32)

## Salida

- `v2_reentrenamiento/resultados_v2_final/scalers/scaler_sutil.joblib`
- `v2_reentrenamiento/resultados_v2_final/scalers/scaler_dulce.joblib`
- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2_escalado.csv`
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2_escalado.csv`

---

## Particiones

| Partición | Rango | Meses |
|---|---|---|
| Train | 2016-07 → 2023-12 | 90 |
| Val (2024) | 2024-01 → 2024-12 | 12 |
| Test (2025) | 2025-01 → 2025-12 | 12 |


## 1. Configuración inicial


In [1]:
import os, warnings
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

warnings.filterwarnings('ignore')
pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

PROC = 'v2_reentrenamiento/data/processed'
OUT  = 'v2_reentrenamiento/resultados_v2_final'
SCALERS = f'{OUT}/scalers'

IN_SUTIL  = f'{PROC}/master_dataset_sutil_v2_features.csv'
IN_DULCE  = f'{PROC}/master_dataset_dulce_v2_features.csv'
OUT_SUTIL = f'{PROC}/master_dataset_sutil_v2_escalado.csv'
OUT_DULCE = f'{PROC}/master_dataset_dulce_v2_escalado.csv'
SCL_SUTIL = f'{SCALERS}/scaler_sutil.joblib'
SCL_DULCE = f'{SCALERS}/scaler_dulce.joblib'

os.makedirs(SCALERS, exist_ok=True)
print('Dir salidas:', SCALERS)


Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
Dir salidas: v2_reentrenamiento/resultados_v2_final/scalers


## 2. Carga y ordenamiento cronológico


In [2]:
master_sutil = pd.read_csv(IN_SUTIL, encoding='utf-8-sig')
master_dulce = pd.read_csv(IN_DULCE, encoding='utf-8-sig')
print('Sutil:', master_sutil.shape, '| Dulce:', master_dulce.shape)

for df in (master_sutil, master_dulce):
    df.sort_values(['año', 'mes'], inplace=True)
    df.reset_index(drop=True, inplace=True)
print('Ordenado OK. Rango:', f'{master_sutil["año"].min()}-{master_sutil["mes"].min()} -> '
      f'{master_sutil["año"].max()}-{master_sutil["mes"].max()}')


Sutil: (114, 32) | Dulce: (114, 32)
Ordenado OK. Rango: 2016-1 -> 2025-12


## 3. Definición del split cronológico

Datos ya libres de NaN (dropna de lags en la Actividad 03). El orden es estrictamente
cronológico: **90 filas train** (2016-07..2023-12), 12 val (2024) y 12 test (2025).


In [3]:
N_TRAIN = 90
N_VAL   = 12
N_TEST  = 12

def particionar(df):
    train = df.iloc[:N_TRAIN].reset_index(drop=True)
    val   = df.iloc[N_TRAIN:N_TRAIN + N_VAL].reset_index(drop=True)
    test  = df.iloc[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST].reset_index(drop=True)
    assert len(train) == 90 and len(val) == 12 and len(test) == 12
    return train, val, test

for nombre, df in [('Sutil', master_sutil), ('Dulce', master_dulce)]:
    tr, va, te = particionar(df)
    print(f'{nombre}: train {tr.shape[0]} filas '
          f'({int(tr.iloc[0].año)}-{int(tr.iloc[0].mes):02d}..{int(tr.iloc[-1].año)}-{int(tr.iloc[-1].mes):02d}) | '
          f'val {va.shape[0]} | test {te.shape[0]}')


Sutil: train 90 filas (2016-07..2023-12) | val 12 | test 12
Dulce: train 90 filas (2016-07..2023-12) | val 12 | test 12


## 4. Variables a escalar

**Se escalan TODAS las columnas numéricas** (producción, precio, clima, lags, INDECI, NLP).

**Se excluyen** por diseño:
- `año`, `mes` — claves de orden/identificación de fila (no son features de entrada del modelo).
- `mes_sin`, `mes_cos` — ya en [-1,1] por la codificación cíclica; escalarlos distorsionaría la
  adyacencia unitaria del círculo trigonométrico.


In [4]:
COL_EXCLUIDAS = ['año', 'mes', 'mes_sin', 'mes_cos']

def cols_escalar(df):
    return [c for c in df.columns if c not in COL_EXCLUIDAS]

CLS_SUTIL = cols_escalar(master_sutil)
CLS_DULCE = cols_escalar(master_dulce)
print(f'Columnas A ESCALAR — SUTIL ({len(CLS_SUTIL)}):')
print(CLS_SUTIL)
print()
print(f'Columnas A ESCALAR — DULCE ({len(CLS_DULCE)}):')
print(CLS_DULCE)


Columnas A ESCALAR — SUTIL (28):
['produccion_t_sutil', 'precio_chacra_kg_sutil', 'n_provincias_sutil', 'T2M', 'T2M_MAX', 'WS2M', 'PRECTOTCORR', 'RH2M', 'num_emergencias', 'personas_afectadas', 'personas_damnificadas', 'total_afectados', 'hectareas_cultivo_perdidas', 'hectareas_cultivo_afectadas', 'avg_sentiment', 'n_noticias', 'produccion_t_sutil_lag1', 'produccion_t_sutil_lag3', 'produccion_t_sutil_lag6', 'T2M_lag1', 'T2M_lag3', 'T2M_lag6', 'WS2M_lag1', 'WS2M_lag3', 'WS2M_lag6', 'PRECTOTCORR_lag1', 'PRECTOTCORR_lag3', 'PRECTOTCORR_lag6']

Columnas A ESCALAR — DULCE (28):
['produccion_t_dulce', 'precio_chacra_kg_dulce', 'n_provincias_dulce', 'T2M', 'T2M_MAX', 'WS2M', 'PRECTOTCORR', 'RH2M', 'num_emergencias', 'personas_afectadas', 'personas_damnificadas', 'total_afectados', 'hectareas_cultivo_perdidas', 'hectareas_cultivo_afectadas', 'avg_sentiment', 'n_noticias', 'produccion_t_dulce_lag1', 'produccion_t_dulce_lag3', 'produccion_t_dulce_lag6', 'T2M_lag1', 'T2M_lag3', 'T2M_lag6', 'WS2M_

In [5]:
# Chequeo: mes_sin/mes_cos deben estar en [-1,1] en TODA la serie
for c in ['mes_sin', 'mes_cos']:
    rng = (master_sutil[c].min(), master_sutil[c].max())
    print(f'{c}: rango en dataset completo = ({rng[0]:+.4f}, {rng[1]:+.4f}) -> '
          f'{"OK [-1,1]" if -1 <= rng[0] and rng[1] <= 1 else "FUERA DE RANGO"}')


mes_sin: rango en dataset completo = (-1.0000, +1.0000) -> OK [-1,1]
mes_cos: rango en dataset completo = (-1.0000, +1.0000) -> OK [-1,1]


## 5. Escalado — fit SOLO sobre train

`StandardScaler.fit(train[COLS_ESCALAR])` y **transform** sobre train, val y test.
Val/test nunca participan en el ajuste (solo se transforman con los parámetros de train).


In [6]:
def escalar(df_cultivo, nombre, cols_escalar, scaler_path):
    train, val, test = particionar(df_cultivo.copy())
    scaler = StandardScaler()
    scaler.fit(train[cols_escalar])  # <- fit SOLO en train (nunca val/test)

    for part in (train, val, test):
        part[cols_escalar] = scaler.transform(part[cols_escalar])

    dataset = pd.concat([train, val, test], ignore_index=True)
    dataset = dataset.sort_values(['año', 'mes']).reset_index(drop=True)
    joblib.dump(scaler, scaler_path)
    print(f'{nombre}: escalado OK -> {dataset.shape} | n_features escaladas={len(cols_escalar)}')
    print(f'Scaller guardado: {scaler_path}')
    return dataset, scaler

ds_sutil, scaler_sutil = escalar(master_sutil, 'SUTIL', CLS_SUTIL, SCL_SUTIL)
ds_dulce, scaler_dulce = escalar(master_dulce, 'DULCE', CLS_DULCE, SCL_DULCE)


SUTIL: escalado OK -> (114, 32) | n_features escaladas=28
Scaller guardado: v2_reentrenamiento/resultados_v2_final/scalers/scaler_sutil.joblib
DULCE: escalado OK -> (114, 32) | n_features escaladas=28
Scaller guardado: v2_reentrenamiento/resultados_v2_final/scalers/scaler_dulce.joblib


## 6. Guardar datasets escalados finales


In [7]:
os.makedirs(PROC, exist_ok=True)
ds_sutil.to_csv(OUT_SUTIL, index=False, encoding='utf-8-sig')
ds_dulce.to_csv(OUT_DULCE, index=False, encoding='utf-8-sig')
print('Guardado:', OUT_SUTIL)
print('Guardado:', OUT_DULCE)


Guardado: v2_reentrenamiento/data/processed/master_dataset_sutil_v2_escalado.csv
Guardado: v2_reentrenamiento/data/processed/master_dataset_dulce_v2_escalado.csv


## 7. Verificación del escalado

1. **Train**: media≈0 y desv≈1 en todas las columnas escaladas.
2. **Val/Test**: la media NO debe ser exactamente 0 (evidencia de que solo se aplicó
   `transform` con parámetros de train, sin `fit`).
3. `mes_sin`/`mes_cos`/`año`/`mes` intactos (no tocados por el scaler).
4. Nulos: 0 en los datasets finales.


In [8]:
def verificar(dataset, nombre):
    cls = cols_escalar(dataset)
    train = dataset.iloc[:N_TRAIN]
    val   = dataset.iloc[N_TRAIN:N_TRAIN + N_VAL]
    test  = dataset.iloc[N_TRAIN + N_VAL:]

    media_train = train[cls].mean()
    std_train   = train[cls].std()
    media_val   = val[cls].mean()
    media_test  = test[cls].mean()

    print('=' * 74)
    print(nombre)
    print('=' * 74)
    print(f'[1] Train (90m): |media| max = {media_train.abs().max():.2e} '
          f'(≈0 OK) | std min = {std_train.min():.4f}, max = {std_train.max():.4f} (≈1 OK)')
    print(f'    Columnas con |media| > 1e-6: {int((media_train.abs() > 1e-6).sum())}')
    print(f'    Columnas con std fuera de [0.99,1.01]: {int(((std_train < 0.99) | (std_train > 1.01)).sum())}')
    print(f'[2] Val (12m): media NO debe ser 0 -> |media| max = {media_val.abs().max():.4f} '
          f'(>0.001 => evidencia transform, no fit)')
    print(f'    min media_val = {media_val.min():+.4f} | max media_val = {media_val.max():+.4f}')
    print(f'    Test(12m): mean range = [{media_test.min():+.4f}, {media_test.max():+.4f}]')
    print(f'[3] Columnas NO escaladas intactas:')
    for c in COL_EXCLUIDAS:
        rng = (dataset[c].min(), dataset[c].max())
        print(f'    {c:10s} rango = ({rng[0]:+.4f}, {rng[1]:+.4f})')
    print(f'[4] Nulos: {int(dataset.isna().sum().sum())} (esperado 0)')
    print()
    ok1 = media_train.abs().max() < 1e-6 and ((std_train < 0.99) | (std_train > 1.01)).sum() == 0
    ok2 = media_val.abs().max() > 0.001
    print('  => VERIFICACIÓN:', 'PASA' if (ok1 and ok2) else 'FALLA')
    return ok1 and ok2

ok_s = verificar(ds_sutil, 'MAESTRO SUTIL v2 ESCALADO')
ok_d = verificar(ds_dulce, 'MAESTRO DULCE v2 ESCALADO')


MAESTRO SUTIL v2 ESCALADO
[1] Train (90m): |media| max = 2.88e-15 (≈0 OK) | std min = 1.0056, max = 1.0056 (≈1 OK)
    Columnas con |media| > 1e-6: 0
    Columnas con std fuera de [0.99,1.01]: 0
[2] Val (12m): media NO debe ser 0 -> |media| max = 1.5222 (>0.001 => evidencia transform, no fit)
    min media_val = -0.5479 | max media_val = +1.5222
    Test(12m): mean range = [-0.2432, +1.4623]
[3] Columnas NO escaladas intactas:
    año        rango = (+2016.0000, +2025.0000)
    mes        rango = (+1.0000, +12.0000)
    mes_sin    rango = (-1.0000, +1.0000)
    mes_cos    rango = (-1.0000, +1.0000)
[4] Nulos: 0 (esperado 0)

  => VERIFICACIÓN: PASA
MAESTRO DULCE v2 ESCALADO
[1] Train (90m): |media| max = 2.88e-15 (≈0 OK) | std min = 1.0056, max = 1.0056 (≈1 OK)
    Columnas con |media| > 1e-6: 0
    Columnas con std fuera de [0.99,1.01]: 0
[2] Val (12m): media NO debe ser 0 -> |media| max = 1.5222 (>0.001 => evidencia transform, no fit)
    min media_val = -0.5479 | max media_val = +1.

### Detalle columna a columna (train vs val/test) — Sutil


In [9]:
train = ds_sutil.iloc[:N_TRAIN]; val = ds_sutil.iloc[N_TRAIN:N_TRAIN+N_VAL]
tabla = pd.DataFrame({
    'media_train': train[CLS_SUTIL].mean().round(4),
    'std_train':   train[CLS_SUTIL].std().round(4),
    'media_val':   val[CLS_SUTIL].mean().round(4),
    'diff |val-0|': val[CLS_SUTIL].mean().abs().round(4),
})
print(tabla.to_string())
print()
print('[2] confirmado: media_val != 0 en la mayoría de features -> scaler NO se ajustó sobre val.')


                             media_train  std_train  media_val  diff |val-0|
produccion_t_sutil                  -0.0     1.0056     1.0914        1.0914
precio_chacra_kg_sutil               0.0     1.0056    -0.1030        0.1030
n_provincias_sutil                   0.0     1.0056     0.1448        0.1448
T2M                                  0.0     1.0056     0.6782        0.6782
T2M_MAX                              0.0     1.0056     0.8715        0.8715
WS2M                                -0.0     1.0056     0.7931        0.7931
PRECTOTCORR                          0.0     1.0056    -0.2401        0.2401
RH2M                                -0.0     1.0056    -0.5479        0.5479
num_emergencias                     -0.0     1.0056     0.2379        0.2379
personas_afectadas                  -0.0     1.0056    -0.1647        0.1647
personas_damnificadas                0.0     1.0056    -0.1817        0.1817
total_afectados                      0.0     1.0056    -0.1756        0.1756

## 8. Validación final e inspección


In [10]:
for nombre, ds in [('SUTIL', ds_sutil), ('DULCE', ds_dulce)]:
    print('=' * 70)
    print(nombre, '|', ds.shape)
    print('-' * 70)
    print('Filas:', len(ds), '(esperado 114)')
    print('Rango:', f'{ds["año"].min()}-{ds["mes"].min()} -> {ds["año"].max()}-{ds["mes"].max()}')
    print('Nulos:', int(ds.isna().sum().sum()), '(esperado 0)')
    print('Columnas:', len(ds.columns))


SUTIL | (114, 32)
----------------------------------------------------------------------
Filas: 114 (esperado 114)
Rango: 2016-1 -> 2025-12
Nulos: 0 (esperado 0)
Columnas: 32
DULCE | (114, 32)
----------------------------------------------------------------------
Filas: 114 (esperado 114)
Rango: 2016-1 -> 2025-12
Nulos: 0 (esperado 0)
Columnas: 32


In [11]:
cols_mostrar = ['año', 'mes', 'mes_sin', 'mes_cos', 'produccion_t_sutil',
                'produccion_t_sutil_lag1', 'T2M', 'avg_sentiment']
print(ds_sutil[cols_mostrar].head(4).round(4).to_string(index=False))
print('...')
print(ds_sutil[cols_mostrar].tail(3).round(4).to_string(index=False))
print()
print('=> Producción (target) escalada con z-scores de train; mes_sin/cos en [-1,1] sin tocar.')
print('=== FIN ACTIVIDAD 04 (v2) — listo para entrenamiento de modelos ===')


 año  mes  mes_sin  mes_cos  produccion_t_sutil  produccion_t_sutil_lag1     T2M  avg_sentiment
2016    7   -0.500   -0.866             -0.5506                  -0.4672 -0.9032         1.1811
2016    8   -0.866   -0.500             -0.3411                  -0.5369 -0.6690         1.2220
2016    9   -1.000   -0.000             -0.6862                  -0.3267 -0.5538         0.7029
2016   10   -0.866    0.500             -0.3606                  -0.6730 -0.6764         1.1861
...
 año  mes  mes_sin  mes_cos  produccion_t_sutil  produccion_t_sutil_lag1     T2M  avg_sentiment
2025   10   -0.866    0.500             -0.2738                  -0.6242 -0.0491         1.1209
2025   11   -0.500    0.866              1.2343                  -0.2592  0.1089         0.7464
2025   12   -0.000    1.000              1.4139                   1.2542  0.2384         0.7865

=> Producción (target) escalada con z-scores de train; mes_sin/cos en [-1,1] sin tocar.
=== FIN ACTIVIDAD 04 (v2) — listo para entr